# Prototype Design Pattern 

explained using the Vector Graphics Editor (Shapes) example (similar to the Memento example, as Prototype is often used to clone complex shapes).

#### The Concept
The Prototype Pattern allows you to create new objects by cloning an existing object (the prototype) rather than creating new instances from scratch using `new Class()`. 

**The Problem**: Imagine you have a complex object `Robot` that takes 5 seconds to configure (fetching data from DB, calculating geometry). If you need 100 identical robots, running the constructor 100 times is slow. The Solution: Create ONE robot (the Prototype), configure it, and then "Clone" it 99 times. Cloning memory is instant.

## The Classic OOP Way (Java-Style)

In strict OOP, we create a `Prototype` interface with a `clone()` method. Every class must implement this method to define exactly how it should be copied (handling references manually).

#### THE PROTOTYPE INTERFACE (Java's "implements Prototype")

In [2]:
from abc import ABC, abstractmethod

class Shape(ABC):
    def __init__(self):
        self.x = 0
        self.y = 0
        self.color = "White"

    @abstractmethod
    def clone(self) -> 'Shape':
        pass

#### CONCRETE PROTOTYPE

In [3]:
class Circle(Shape):
    def __init__(self):
        super().__init__()
        self.radius = 0

    def clone(self) -> 'Circle':
        # Create new instance
        new_circle = Circle()
        # Copy data manually
        new_circle.x = self.x
        new_circle.y = self.y
        new_circle.color = self.color
        new_circle.radius = self.radius
        return new_circle

    def __str__(self):
        return f"Circle(x={self.x}, y={self.y}, c={self.color}, r={self.radius})"

class Rectangle(Shape):
    def __init__(self):
        super().__init__()
        self.width = 0
        self.height = 0

    def clone(self) -> 'Rectangle':
        new_rect = Rectangle()
        new_rect.x = self.x
        new_rect.y = self.y
        new_rect.color = self.color
        new_rect.width = self.width
        new_rect.height = self.height
        return new_rect

    def __str__(self):
        return f"Rect(x={self.x}, y={self.y}, c={self.color}, w={self.width})"

### CLIENT CODE

In [4]:
def main():
    print("--- Java-Style Prototype ---")
    
    # 1. Configure the "Master" object (Expensive setup)
    master_circle = Circle()
    master_circle.x = 10
    master_circle.y = 10
    master_circle.radius = 20
    master_circle.color = "Red"
    
    print(f"Original: {master_circle}")

    # 2. Clone it (Cheap)
    clone_1 = master_circle.clone()
    clone_1.x = 99 # Change just one thing
    
    print(f"Clone:    {clone_1}")
    print(f"Original remains: {master_circle}")

if __name__ == "__main__":
    main()

--- Java-Style Prototype ---
Original: Circle(x=10, y=10, c=Red, r=20)
Clone:    Circle(x=99, y=10, c=Red, r=20)
Original remains: Circle(x=10, y=10, c=Red, r=20)


#### Why this feels like Java

- Explicit Interface: We use ABC to strictly enforce the clone method, similar to implementing an interface in Java.
- Explicit Constructor: We use __init__ with explicit assignments (self.x = x) instead of Python's dataclass shortcut.
- Strict Method Definition: The clone method is explicitly defined in the class, wrapping the internal copy logic.

## Prototype Design Pattern (Pythonic Way)

Python has the Prototype Pattern built-in. We do not need to write a clone() method. We simply use the standard copy module.
- `copy.copy()`: Shallow copy (references are shared).
- `copy.deepcopy()`: Deep copy (everything is duplicated recursively).

#### The Concept
The Prototype pattern is used when creating a new object from scratch is expensive (e.g., database queries, complex calculations). Instead of building a new object, you `clone` (copy) an existing one and modify it slightly.

Think of it like a `"Save As..."` feature in a document editor. You don't type the whole document again; you copy the old one and change the title.

The Code (Pythonic Way)
In Python, we don't need a complex `Cloneable` interface like in Java. We simply use the built-in `copy` module.

#### THE CLASSES (Just Data)

In [7]:
import copy
from dataclasses import dataclass, field
from typing import List

@dataclass
class Robot:
    name: str
    model: str
    skills: List[str] = field(default_factory=list)
    battery_level: int = 100

    def clone(self) -> "Robot":
        """
        Creates a deep copy of the robot.
        We use deepcopy to ensure that modifying the 'skills' list
        of the clone does NOT affect the original robot.
        """
        return copy.deepcopy(self)

    def __str__(self):
        return (f"🤖 [{self.name}] Model: {self.model} | "
                f"Battery: {self.battery_level}% | Skills: {self.skills}")

#### CLIENT CODE

In [8]:
def main():
    # 1. Create a "Base" Prototype (Expensive operation simulation)
    # Imagine this step involves downloading AI models or heavy config
    print("--- Creating Prototype ---")
    base_robot = Robot(name="Prototype-01", model="T-800")
    base_robot.skills.append("Walk")
    base_robot.skills.append("Talk")
    
    print(f"Original: {base_robot}\n")

    # 2. Clone the Prototype to create a new Robot
    # This is much faster than setting up a new robot from scratch
    print("--- Cloning Robot ---")
    worker_robot = base_robot.clone()
    
    # 3. Customize the Clone
    worker_robot.name = "Worker-Bot-A"
    worker_robot.skills.append("Weld") # Add a new skill only for this clone
    
    # 4. Create another Clone (Fighter)
    fighter_robot = base_robot.clone()
    fighter_robot.name = "Fighter-Bot-X"
    fighter_robot.skills.append("Fight")
    fighter_robot.model = "T-1000"

    # ==========================================
    # 3. VERIFICATION
    # ==========================================
    print(f"Clone 1:  {worker_robot}")
    print(f"Clone 2:  {fighter_robot}")
    
    # Verify the Original is untouched (Deep Copy worked)
    print(f"\nOriginal Integrity Check: {base_robot}")
    # If we didn't use deepcopy, 'Weld' and 'Fight' would have appeared here!

if __name__ == "__main__":
    main()

--- Creating Prototype ---
Original: 🤖 [Prototype-01] Model: T-800 | Battery: 100% | Skills: ['Walk', 'Talk']

--- Cloning Robot ---
Clone 1:  🤖 [Worker-Bot-A] Model: T-800 | Battery: 100% | Skills: ['Walk', 'Talk', 'Weld']
Clone 2:  🤖 [Fighter-Bot-X] Model: T-1000 | Battery: 100% | Skills: ['Walk', 'Talk', 'Fight']

Original Integrity Check: 🤖 [Prototype-01] Model: T-800 | Battery: 100% | Skills: ['Walk', 'Talk']


#### Why copy.deepcopy()?

If your object contains lists or other objects (like skills: List[str]), a normal copy (shallow copy) only copies the reference to the list.
- Shallow Copy: If you add "Weld" to the clone's skill list, the original robot also learns to weld. (Bad!)
- Deep Copy: The clone gets a completely new list. Changes don't affect the original.

#### When to use this?

- Performance: When __init__ is very slow (e.g., it connects to a database or parses a huge XML file).
- Configuration: When you have a complex default configuration (like a "Standard User Permission Set") and you want to create a new user by just tweaking the standard set.


# Prototype Design Pattern 

explained using a complex, real-world software industry example: Game Development (Entity Spawning System).

#### The Scenario: Spawning Hordes of Enemies

In a game like "Left 4 Dead" or an RTS like "StarCraft," the game engine needs to spawn hundreds of enemies (Zombies, Orcs) instantly.
- The Problem: Loading an enemy from scratch is "Heavy". It involves:
- Loading the 3D Mesh from the hard drive.
- Loading Textures.
- Parsing AI scripts.
- Setting up physics collision boxes. If you run `new Zombie()` 100 times, the game will lag because it repeats these heavy steps.

**The Solution**: You create one "Master Zombie" (The Prototype) with everything loaded. When you need a horde, you simply Clone the master in memory. It's strictly a memory copy operation (very fast).

## The Classic OOP Way (Java-Style)

We define an Enemy interface with a `clone` method. The concrete classes (`Zombie`, `Orc`) manage their own complex data copying (like lists of weapons).

#### THE PROTOTYPE INTERFACE

In [9]:
from abc import ABC, abstractmethod

class Enemy(ABC):
    def __init__(self):
        self.health = 100
        self.speed = 10
        self.location = (0,0,0)
        # Simulating heavy resource
        self.mesh_data = "HEAVY_3D_MODEL_DATA_LOADED_FROM_DISK" 
        self.weapons: List[str] = []

    @abstractmethod
    def clone(self) -> 'Enemy':
        pass

    def equip_weapon(self, weapon):
        self.weapons.append(weapon)

    def __str__(self):
        return (f"[{self.__class__.__name__}] HP:{self.health} "
                f"Weapons:{self.weapons} Mesh:{len(self.mesh_data)} bytes")

#### CONCRETE PROTOTYPES

In [11]:
import copy

class Zombie(Enemy):
    def __init__(self):
        super().__init__()
        self.type = "Walker"

    def clone(self) -> 'Zombie':
        # 1. Create new instance (bypassing heavy init if possible)
        new_zombie = Zombie()
        
        # 2. Copy primitives (Fast)
        new_zombie.health = self.health
        new_zombie.speed = self.speed
        new_zombie.type = self.type
        
        # 3. Share heavy immutable resources (Optimization!)
        # We don't copy the mesh string; we point to the same one.
        new_zombie.mesh_data = self.mesh_data
        
        # 4. Deep Copy mutable lists (Crucial!)
        # If we don't do this, adding a weapon to one zombie adds it to all.
        new_zombie.weapons = copy.deepcopy(self.weapons)
        
        return new_zombie

class Orc(Enemy):
    def __init__(self):
        super().__init__()
        self.armor = 50

    def clone(self) -> 'Orc':
        new_orc = Orc()
        new_orc.health = self.health
        new_orc.mesh_data = self.mesh_data # Shared
        new_orc.weapons = copy.deepcopy(self.weapons)
        new_orc.armor = self.armor
        return new_orc

#### CLIENT CODE (The Spawner)

In [12]:
class EnemySpawner:
    def __init__(self, prototype: Enemy):
        self.prototype = prototype

    def spawn_horde(self, count: int):
        horde = []
        for _ in range(count):
            # Cloning is much faster than re-initializing
            clone = self.prototype.clone()
            horde.append(clone)
        return horde

def main():
    print("--- Java-Style Game Spawner ---")

    # 1. Setup the Master Prototype (Expensive Step)
    master_zombie = Zombie()
    master_zombie.equip_weapon("Claws")
    master_zombie.health = 150 # Custom "Tank" Zombie

    # 2. Create Spawner
    spawner = EnemySpawner(master_zombie)

    # 3. Spawn!
    horde = spawner.spawn_horde(3)
    
    # 4. Prove Independence
    horde[0].equip_weapon("Infection Bile") # Modify only the first one

    for z in horde:
        print(z)

if __name__ == "__main__":
    main()

--- Java-Style Game Spawner ---
[Zombie] HP:150 Weapons:['Claws', 'Infection Bile'] Mesh:36 bytes
[Zombie] HP:150 Weapons:['Claws'] Mesh:36 bytes
[Zombie] HP:150 Weapons:['Claws'] Mesh:36 bytes


## The Pythonic Way (Registry & Deepcopy)

In a real Python game engine (or data science pipeline), we don't write manual clone methods for every entity. Instead, we use a **Prototype Registry**. This is a dictionary that holds pre-configured objects. We use `copy.deepcopy` to handle the cloning logic automatically.

This pattern is extremely common in **Configuration Systems**: "Give me a default database config, but change the port."

#### THE ENTITY (Pure Data)

In [15]:
from dataclasses import dataclass, field
from typing import Dict, List

@dataclass
class GameEntity:
    name: str
    health: int
    assets: Dict[str, str] = field(default_factory=dict) # Heavy data
    buffs: List[str] = field(default_factory=list)       # Mutable list

    def __str__(self):
        return f"👾 {self.name} (HP:{self.health}) | Buffs: {self.buffs}"

#### THE PROTOTYPE REGISTRY (The Factory)

In [16]:
from typing import Dict
import copy

class EntityFactory:
    """
    Stores prototypes and serves clones.
    """
    _prototypes: Dict[str, GameEntity] = {}

    @classmethod
    def register(cls, key: str, entity: GameEntity):
        cls._prototypes[key] = entity

    @classmethod
    def create(cls, key: str, **overrides) -> GameEntity:
        """
        1. Find the prototype.
        2. Deep Copy it.
        3. Apply overrides (optional customization).
        """
        if key not in cls._prototypes:
            raise ValueError(f"Unknown entity: {key}")
        
        # THE CORE PATTERN: Deep Copy
        clone = copy.deepcopy(cls._prototypes[key])
        
        # Pythonic convenience: Update attributes dynamically
        for k, v in overrides.items():
            if hasattr(clone, k):
                setattr(clone, k, v)
                
        return clone

#### CLIENT CODE

In [18]:
def main():
    print("--- Pythonic Entity Spawner ---")

    # 1. Initialize Prototypes (Do this once at level load)
    # Standard Goblin
    goblin_grunt = GameEntity("Goblin Grunt", 50, assets={"mesh": "goblin.obj"}, buffs=["Speed"])
    # Boss Goblin
    goblin_king = GameEntity("Goblin King", 500, assets={"mesh": "king.obj"}, buffs=["Aura", "Armor"])

    # 2. Register them
    EntityFactory.register("grunt", goblin_grunt)
    EntityFactory.register("boss", goblin_king)

    # 3. Gameplay Loop: Spawn Entities
    
    # Spawn a standard grunt
    enemy1 = EntityFactory.create("grunt")
    
    # Spawn a variant (e.g., a grunt that leveled up)
    # We use overrides to customize the clone immediately
    enemy2 = EntityFactory.create("grunt", health=80, name="Veteran Grunt")
    enemy2.buffs.append("Shield") # Modify mutable list

    # Spawn the boss
    boss = EntityFactory.create("boss")

    # 4. Verify Independence
    print(enemy1) # Should have standard HP and Buffs
    print(enemy2) # Should have higher HP and extra Shield
    print(boss)

    # Proof that heavy assets are copied correctly
    print(f"\nIndependence Check: enemy1 assets: {id(enemy1.assets)} vs enemy2 assets: {id(enemy2.assets)}")
    # Note: deepcopy creates new dictionaries for assets. 
    # If assets were truly read-only heavy data (like pointers to C++ memory), 
    # we would implement __deepcopy__ to prevent copying them, for optimization.

if __name__ == "__main__":
    main()

--- Pythonic Entity Spawner ---
👾 Goblin Grunt (HP:50) | Buffs: ['Speed']
👾 Veteran Grunt (HP:80) | Buffs: ['Speed', 'Shield']
👾 Goblin King (HP:500) | Buffs: ['Aura', 'Armor']

Independence Check: enemy1 assets: 264740381123776 vs enemy2 assets: 264740381125696


#### Why the Pythonic version is better here

- **Registry Pattern**: Instead of passing a prototype instance to a spawner class (`EnemySpawner(master_zombie)`), we have a central `EntityFactory` that acts as a catalog. This allows any part of the code to say `EntityFactory.create("grunt")`.
- `**overrides`: The ability to pass `health=80` into the `create` method is powerful. It combines the **Prototype** pattern with the **Builder** pattern behavior: "Give me a copy of X, but with these specific changes."
- **Maintenance**: If you add a new field `mana` to `GameEntity`, `copy.deepcopy` handles it automatically. In the Java version, you'd have to manually update the `clone()` method to ensure `new_zombie.mana = self.mana`.